In [1]:
# Phase 2: Data Preprocessing

# Prepare the data for modeling by handling missing values, defining the target variable, and selecting relevant features.

In [2]:
import pandas as pd
import numpy as np
import os

# Adjust display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported.")

Libraries imported.


In [8]:
# Define the path to the raw data file
# Assuming notebook is in 'notebooks/' dir
data_file_root = '../flights_sample_3m.csv'
data_file_data_dir = '../data/flights_sample_3m.csv'

if os.path.exists(data_file_root):
    data_path = data_file_root
elif os.path.exists(data_file_data_dir):
    data_path = data_file_data_dir
else:
    # Try checking relative to current dir (if running notebook from root)
    data_file_root_alt = 'flights_sample_3m.csv'
    data_file_data_dir_alt = 'data/flights_sample_3m.csv'
    if os.path.exists(data_file_root_alt):
        data_path = data_file_root_alt
    elif os.path.exists(data_file_data_dir_alt):
        data_path = data_file_data_dir_alt
    else:
        raise FileNotFoundError("Could not find flights_sample_3m.csv in root or data/ directory.")

print(f"Loading raw data from: {data_path}")

# Load the dataset - load only necessary columns to save memory
# Columns needed for target + features + intermediate checks
cols_to_load = [
    'FL_DATE', 'AIRLINE', 'ORIGIN', 'DEST', 'DISTANCE',
    'CRS_DEP_TIME', 'CRS_ARR_TIME', 'CRS_ELAPSED_TIME',
    'ARR_DELAY', 'CANCELLED', 'DIVERTED' # Load ARR_DELAY, CANCELLED, DIVERTED for target definition
]
# Check if all these columns exist in the actual file first
try:
    df_peek = pd.read_csv(data_path, nrows=0) # Read only header
    existing_cols_to_load = [col for col in cols_to_load if col in df_peek.columns]
    print(f"Columns found and will be loaded: {existing_cols_to_load}")
    if set(cols_to_load) != set(existing_cols_to_load):
        print(f"Warning: Could not find columns: {list(set(cols_to_load) - set(existing_cols_to_load))}")

    df_raw = pd.read_csv(data_path, usecols=existing_cols_to_load)
except Exception as e:
    print(f"Error loading specific columns: {e}. Loading all columns instead.")
    df_raw = pd.read_csv(data_path)


print("Raw dataset loaded successfully.")
print(f"Shape: {df_raw.shape}")

# Display memory usage
df_raw.info(memory_usage='deep')

Loading raw data from: ../data/flights_sample_3m.csv
Columns found and will be loaded: ['FL_DATE', 'AIRLINE', 'ORIGIN', 'DEST', 'DISTANCE', 'CRS_DEP_TIME', 'CRS_ARR_TIME', 'CRS_ELAPSED_TIME', 'ARR_DELAY', 'CANCELLED', 'DIVERTED']
Raw dataset loaded successfully.
Shape: (3000000, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000000 entries, 0 to 2999999
Data columns (total 11 columns):
 #   Column            Dtype  
---  ------            -----  
 0   FL_DATE           object 
 1   AIRLINE           object 
 2   ORIGIN            object 
 3   DEST              object 
 4   CRS_DEP_TIME      int64  
 5   CRS_ARR_TIME      int64  
 6   ARR_DELAY         float64
 7   CANCELLED         float64
 8   DIVERTED          float64
 9   CRS_ELAPSED_TIME  float64
 10  DISTANCE          float64
dtypes: float64(5), int64(2), object(4)
memory usage: 822.8 MB


In [9]:
print("Original shape:", df_raw.shape)

# Ensure ARR_DELAY is numeric, coercing errors to NaN
if 'ARR_DELAY' in df_raw.columns:
    df_raw['ARR_DELAY'] = pd.to_numeric(df_raw['ARR_DELAY'], errors='coerce')
else:
    raise KeyError("ARR_DELAY column not found, cannot define target variable.")

# Check for CANCELLED and DIVERTED columns and filter
initial_rows = len(df_raw)
rows_dropped = 0

if 'CANCELLED' in df_raw.columns:
    cancelled_count = df_raw['CANCELLED'].sum()
    print(f"Filtering out {cancelled_count} cancelled flights...")
    df_processed = df_raw[df_raw['CANCELLED'] == 0].copy()
    rows_dropped += initial_rows - len(df_processed)
    initial_rows = len(df_processed)
else:
    print("CANCELLED column not found, skipping cancellation filter.")
    df_processed = df_raw.copy() # Start with a copy

if 'DIVERTED' in df_processed.columns:
    diverted_count = df_processed['DIVERTED'].sum()
    print(f"Filtering out {diverted_count} diverted flights...")
    df_processed = df_processed[df_processed['DIVERTED'] == 0]
    rows_dropped += initial_rows - len(df_processed)
    initial_rows = len(df_processed)
else:
    print("DIVERTED column not found, skipping diverted filter.")

# Now handle missing ARR_DELAY - after filtering cancellations/diversions,
# remaining NaNs might be other issues, let's drop them for target definition.
missing_arr_delay_before = df_processed['ARR_DELAY'].isnull().sum()
if missing_arr_delay_before > 0:
    print(f"Dropping {missing_arr_delay_before} rows with missing ARR_DELAY after filtering...")
    df_processed.dropna(subset=['ARR_DELAY'], inplace=True)
    rows_dropped += initial_rows - len(df_processed)

# Define the target variable: 1 if ARR_DELAY > 15, 0 otherwise
delay_threshold = 15
df_processed['IS_DELAYED'] = (df_processed['ARR_DELAY'] > delay_threshold).astype(int)

print(f"\nTotal rows dropped (Cancelled/Diverted/Missing Target): {rows_dropped}")
print("Shape after filtering and target definition:", df_processed.shape)
print("\nTarget variable distribution:")
print(df_processed['IS_DELAYED'].value_counts(normalize=True))

# Display sample of rows with the new target
display(df_processed[['FL_DATE', 'ARR_DELAY', 'IS_DELAYED']].head())

Original shape: (3000000, 11)
Filtering out 79140.0 cancelled flights...
Filtering out 7056.0 diverted flights...
Dropping 2 rows with missing ARR_DELAY after filtering...

Total rows dropped (Cancelled/Diverted/Missing Target): 86198
Shape after filtering and target definition: (2913802, 12)

Target variable distribution:
IS_DELAYED
0    0.823156
1    0.176844
Name: proportion, dtype: float64


,FL_DATE,ARR_DELAY,IS_DELAYED
0,2019-01-09,-14.0,0
1,2022-11-19,-5.0,0
2,2022-07-22,0.0,0
3,2023-03-06,24.0,1
4,2020-02-23,-1.0,0


In [11]:
# Define columns to keep (features + target)
# These are known before the flight departs
features_to_keep = [
    'FL_DATE',
    'AIRLINE', # <--- MAKE SURE THIS IS PRESENT
    'ORIGIN',
    'DEST',
    'DISTANCE',
    'CRS_DEP_TIME',
    'CRS_ARR_TIME',
    'CRS_ELAPSED_TIME',
    'IS_DELAYED'  # Our target variable
]

# Identify columns in our current dataframe to keep
actual_cols_to_keep = [col for col in features_to_keep if col in df_processed.columns]

print(f"Columns to keep: {actual_cols_to_keep}")

# Select only these columns
df_selected = df_processed[actual_cols_to_keep].copy()

# Drop the original columns used for filtering/target definition if they weren't explicitly kept
# (e.g., CANCELLED, DIVERTED, ARR_DELAY) - though they should be gone if not in features_to_keep
cols_to_drop_explicit = ['CANCELLED', 'DIVERTED', 'ARR_DELAY']
for col in cols_to_drop_explicit:
    if col in df_selected.columns and col not in actual_cols_to_keep:
        df_selected = df_selected.drop(columns=[col])


print("\nShape after selecting features:", df_selected.shape)
print("Columns remaining:", df_selected.columns.tolist())

# Verify data types again
print("\nData types of selected features:")
df_selected.info()

Columns to keep: ['FL_DATE', 'AIRLINE', 'ORIGIN', 'DEST', 'DISTANCE', 'CRS_DEP_TIME', 'CRS_ARR_TIME', 'CRS_ELAPSED_TIME', 'IS_DELAYED']

Shape after selecting features: (2913802, 9)
Columns remaining: ['FL_DATE', 'AIRLINE', 'ORIGIN', 'DEST', 'DISTANCE', 'CRS_DEP_TIME', 'CRS_ARR_TIME', 'CRS_ELAPSED_TIME', 'IS_DELAYED']

Data types of selected features:
<class 'pandas.core.frame.DataFrame'>
Index: 2913802 entries, 0 to 2999999
Data columns (total 9 columns):
 #   Column            Dtype  
---  ------            -----  
 0   FL_DATE           object 
 1   AIRLINE           object 
 2   ORIGIN            object 
 3   DEST              object 
 4   DISTANCE          float64
 5   CRS_DEP_TIME      int64  
 6   CRS_ARR_TIME      int64  
 7   CRS_ELAPSED_TIME  float64
 8   IS_DELAYED        int32  
dtypes: float64(2), int32(1), int64(2), object(4)
memory usage: 211.2+ MB


In [12]:
# Handle the few missing values in CRS_ELAPSED_TIME
if 'CRS_ELAPSED_TIME' in df_selected.columns:
    missing_crs_time = df_selected['CRS_ELAPSED_TIME'].isnull().sum()
    print(f"\nMissing CRS_ELAPSED_TIME before imputation: {missing_crs_time}")

    if missing_crs_time > 0:
        # Impute with the median (less sensitive to outliers than mean)
        median_crs_time = df_selected['CRS_ELAPSED_TIME'].median()
        df_selected['CRS_ELAPSED_TIME'].fillna(median_crs_time, inplace=True)
        print(f"Imputed missing CRS_ELAPSED_TIME with median value: {median_crs_time}")

    # Verify imputation
    missing_crs_time_after = df_selected['CRS_ELAPSED_TIME'].isnull().sum()
    print(f"Missing CRS_ELAPSED_TIME after imputation: {missing_crs_time_after}")
else:
    print("\nCRS_ELAPSED_TIME column not found in selected features.")

# Final check for any remaining missing values in the selected dataframe
print("\nFinal check for missing values in df_selected:")
print(df_selected.isnull().sum())



Missing CRS_ELAPSED_TIME before imputation: 0
Missing CRS_ELAPSED_TIME after imputation: 0

Final check for missing values in df_selected:
FL_DATE             0
AIRLINE             0
ORIGIN              0
DEST                0
DISTANCE            0
CRS_DEP_TIME        0
CRS_ARR_TIME        0
CRS_ELAPSED_TIME    0
IS_DELAYED          0
dtype: int64


In [13]:
# Define path for processed data
# Create data directory if it doesn't exist
output_dir = '../data'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")

output_file = os.path.join(output_dir, 'flights_processed_v1.csv')

print(f"\nSaving processed data to: {output_file} ...")
# Use index=False to avoid writing the dataframe index as a column
df_selected.to_csv(output_file, index=False)

print("Processed data saved successfully.")
print(f"Final shape saved: {df_selected.shape}")

# Optional: Verify file size
file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
print(f"Saved file size: {file_size_mb:.2f} MB")


Saving processed data to: ../data\flights_processed_v1.csv ...
Processed data saved successfully.
Final shape saved: (2913802, 9)
Saved file size: 177.97 MB
